Please copy this collab notebook in order to edit!

Please make sure that your batch information is labelled as "batch". If you are not sure how to do this, please follow the steps in the notebook [here](https://colab.research.google.com/drive/11a-QpqPnBFvdB3ySQSZuoICXvuWKxQwm?usp=sharing) to label your batch info.

Your batched results will be saved as "batch_i.h5ad", where "i" is the ith batch file.

Once you have finished editing, select Runtime -> Run all in the top left.

In [ ]:
path_to_file = "test_data/HLCA_delorey.h5ad" #please change file name in quotations to input .h5ad file

In [ ]:
!pip install scanpy

In [ ]:
import scanpy as sc
adata = sc.read(path_to_file)

In [ ]:
batch_label="batch"

In [ ]:
batches=adata.obs[batch_label].value_counts().sort_values(ascending=False)
batches_to_try=batches.copy()

In [ ]:
#check if any batch is larger than 200 000
batches_gt_thresh = batches[batches.values>200000].index
count=1
if len(batches_gt_thresh)>0:
    for batch in batches_gt_thresh:
        adata[adata.obs[batch_label] == batch].write(f"batch_{count}.h5ad")
        count+=1
    batches.drop(batches_gt_thresh, inplace=True)

In [ ]:
# Combine remaining batches into groups <= 200,000 cells
if not batches.empty:
    # Sort descending by size
    batches = batches.sort_values(ascending=False)
    groups = []
    current_group = []
    current_size = 0
    for batch, size in batches.items():
        if current_size + size <= 200000:
            current_group.append(batch)
            current_size += size
        else:
            if current_group:
                groups.append(current_group)
            current_group = [batch]
            current_size = size
    if current_group:
        groups.append(current_group)
    
    # Save each group
    for group in groups:
        adata[adata.obs[batch_label].isin(group)].write(f"batch_{count}.h5ad")
        count += 1